In [2]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as mticker
import pandas as pd
import seaborn as sns
from scipy.stats import gaussian_kde, wasserstein_distance, ks_2samp, pearsonr, spearmanr
import glob
import os
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
from sklearn.manifold import trustworthiness
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
import umap
from scipy.stats import mannwhitneyu
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
import os
from scipy.spatial import cKDTree
from scipy.sparse.csgraph import connected_components
import matplotlib.colors as mcolors


In [3]:
from picosvg.svg import SVG

def save_flat_svg(fig, path, dpi=500):
    fig.savefig(path, dpi=dpi)
    svg = SVG.parse(path).topicosvg()
    with open(path, "w") as f:
        f.write(svg.tostring())

In [4]:
plt.rcParams.update({
    "font.size": 18,
    "axes.labelsize": 17,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
})

# Figure Panel #6

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

HIST_COLOR = "#808080"
KDE_COLOR = "black"
PDPT_HIST_COLOR = "#4C72B0"
PDPT_KDE_COLOR = "#1f3d6b"
TARGET_COLOR = "black"

plt.rcParams.update({"font.size": 14})


def save_flat_svg(fig, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    fig.savefig(path, format="svg", dpi=500, bbox_inches="tight")
    print(f"Saved plot to {path}")


def load_binding_energies(file):
    df = pd.read_csv(file)
    if "computed_binding_E" not in df.columns:
        raise KeyError("Expected column 'computed_binding_E' not found in CSV")
    energies = df["computed_binding_E"].dropna().to_numpy()
    if energies.size == 0:
        raise ValueError("No valid (non-NaN) computed_binding_E values found")
    return energies, df


def load_pdpt_energies(file):
    df = pd.read_csv(file)
    energies = df["Binding_Energy_eV"].dropna().to_numpy()
    return energies


def plot_distribution(energies, pdpt_energies, target_E, target_tol, output_path):
    fig, ax = plt.subplots(figsize=(8, 5), dpi=500)

    n_bins = min(30, max(10, int(np.sqrt(len(energies)))))
    counts, bin_edges, _ = ax.hist(
        energies, bins=n_bins, density=False,
        color=HIST_COLOR, alpha=0.6, edgecolor="white", label="Binding energy"
    )
    bin_width = bin_edges[1] - bin_edges[0]
    if len(energies) > 1:
        kde = gaussian_kde(energies)
        x = np.linspace(energies.min(), energies.max(), 300)
        ax.plot(x, kde(x) * len(energies) * bin_width, color=KDE_COLOR, linewidth=2, label="KDE")

    n_bins_pdpt = min(30, max(10, int(np.sqrt(len(pdpt_energies)))))
    counts_pdpt, bin_edges_pdpt, _ = ax.hist(
        pdpt_energies, bins=n_bins_pdpt, density=False,
        color=PDPT_HIST_COLOR, alpha=0.6, edgecolor="white", label="Pd/Pt binding energy"
    )
    bin_width_pdpt = bin_edges_pdpt[1] - bin_edges_pdpt[0]
    if len(pdpt_energies) > 1:
        kde_pdpt = gaussian_kde(pdpt_energies)
        x_pdpt = np.linspace(pdpt_energies.min(), pdpt_energies.max(), 300)
        ax.plot(x_pdpt, kde_pdpt(x_pdpt) * len(pdpt_energies) * bin_width_pdpt,
                 color=PDPT_KDE_COLOR, linewidth=2, label="Pd/Pt KDE")

    ax.axvline(target_E, color=TARGET_COLOR, linestyle="--", linewidth=1.4, zorder=5)

    ax.set_xlabel("Computed binding energy (eV)")
    ax.set_ylabel("Count")
    ax.set_xlim(-4, 4)
    ax.set_ylim(0, 4000)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=14)

    fig.tight_layout()
    save_flat_svg(fig, output_path)
    plt.show()


file = "/Users/deanmsweeney/Documents/CFM/gace/manuscript/data/H_volcano/relaxed_strucs_data.csv"
pdpt_file = os.path.join(os.path.dirname(file), "pd_pt_data.csv")
output_path = "./fig5/panel/binding_energy_dist.svg"
target_E = -0.27
target_tol = 0.20

energies, df = load_binding_energies(file)
pdpt_energies = load_pdpt_energies(pdpt_file)
plot_distribution(energies, pdpt_energies, target_E, target_tol, output_path)

In [ ]:
"./data/H_volcano/H_element_count.csv"

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

POINT_COLOR = "#4C72B0"
POINT_EDGE = "black"
TARGET_COLOR = "black"
ZERO_LINE_COLOR = "#808080"

plt.rcParams.update({"font.size": 14})


def save_flat_svg(fig, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    fig.savefig(path, format="svg", dpi=500, bbox_inches="tight")
    print(f"Saved plot to {path}")


def load_element_binding(file):
    df = pd.read_csv(file)
    required = {"Element", "Count", "Avg_Binding_Energy_eV", "N_With_Energy"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing expected column(s): {missing}")
    df = df.dropna(subset=["Avg_Binding_Energy_eV"])
    if df.empty:
        raise ValueError("No valid (non-NaN) Avg_Binding_Energy_eV rows found")
    return df.sort_values("Avg_Binding_Energy_eV")


def plot_element_binding(df, target_E=None, output_path=None):
    fig, ax = plt.subplots(figsize=(6, max(8, len(df) * 0.22)), dpi=500)

    y = range(len(df))
    sizes = 15 + 60 * (df["Count"] / df["Count"].max())

    ax.scatter(df["Avg_Binding_Energy_eV"], y, s=sizes, color=POINT_COLOR,
               edgecolor=POINT_EDGE, linewidth=0.4, zorder=3)

    ax.set_yticks(y)
    ax.set_yticklabels(df["Element"])
    ax.set_xlabel("Average binding energy (eV)")
    ax.set_ylabel("Element")
    ax.axvline(0, color=ZERO_LINE_COLOR, linewidth=0.8, zorder=1)

    if target_E is not None:
        ax.axvline(target_E, color=TARGET_COLOR, linestyle="--", linewidth=1.4,
                   zorder=2, label=f"Target = {target_E} eV")
        ax.legend(frameon=False, fontsize=12, loc="lower right")

    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    if output_path:
        save_flat_svg(fig, output_path)
    plt.show()

    return fig, ax


def plot_top20_by_distance(df, target_E, top_n=20, output_path=None):
    top = df.nlargest(top_n, "Count").copy()
    top["distance"] = (top["Avg_Binding_Energy_eV"] - target_E).abs()
    top = top.sort_values("distance")

    fig, ax = plt.subplots(figsize=(8, 6), dpi=500)

    x = range(len(top))
    bars = ax.bar(x, top["Avg_Binding_Energy_eV"], color=plt.cm.viridis_r(
        top["distance"] / top["distance"].max()), edgecolor="black", linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(top["Element"], rotation=45, ha="right")
    ax.set_ylabel("Average binding energy (eV)")
    ax.set_xlabel(f"Element (top {top_n} by count)")
    ax.axhline(target_E, color=TARGET_COLOR, linestyle="--", linewidth=1.4,
               zorder=2, label=f"Target = {target_E} eV")
    ax.axhline(0, color=ZERO_LINE_COLOR, linewidth=0.8, zorder=1)
    ax.legend(frameon=False, fontsize=12)
    ax.spines[["top", "right"]].set_visible(False)

    sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis_r,
                                norm=plt.Normalize(0, top["distance"].max()))
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label("|Distance from target| (eV)")

    fig.tight_layout()

    if output_path:
        save_flat_svg(fig, output_path)
    plt.show()

    return fig, ax


file = "./data/H_volcano/H_element_count.csv"
output_path = "./fig5/panel/element_binding_energy.svg"
top20_output_path = "./fig5/panel/top20_element_distance.svg"
target_E = -0.3

df = load_element_binding(file)
plot_element_binding(df, target_E=target_E, output_path=output_path)
plot_top20_by_distance(df, target_E=target_E, output_path=top20_output_path)

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

LINE_COLOR = "black"
REGION_COLOR = "red"
HIST_COLOR = "#808080"
KDE_COLOR = "black"
DEFAULT_COLOR = "#B0B0B0"

# Elements to highlight, in priority order (first match wins if a formula has more than one)
HIGHLIGHT_ELEMENTS = ["Ti", "Al", "Rh", "Ru", "Co", "Fe", "Ni", "Cu"]
HIGHLIGHT_COLORS = {
    "Ti": "#4C72B0",
    "Al": "#DD8452",
    "Rh": "#55A868",
    "Ru": "#C44E52",
    "Co": "#8172B2",
    "Fe": "#937860",
    "Ni": "#DA8BC3",
    "Cu": "#CCB974",
}

# Original high-activity lines from paper (N_ads = slope * O_ads + intercept, paper reference)
LEFT_SLOPE, LEFT_INT = 1.91, 5.47
RIGHT_SLOPE, RIGHT_INT = 1.70, 1.65
N_SHIFT = 5.189
O_SHIFT = 2.846

plt.rcParams.update({"font.size": 14})


def save_flat_svg(fig, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    fig.savefig(path, format="svg", dpi=500, bbox_inches="tight")
    print(f"Saved plot to {path}")


def load_binding_energies(file):
    df = pd.read_csv(file)
    for col in ("n_computed_binding_E", "o_computed_binding_E", "reduced_formula"):
        if col not in df.columns:
            raise KeyError(f"Expected column '{col}' not found in CSV")
    df = df.dropna(subset=["n_computed_binding_E", "o_computed_binding_E"])
    if df.empty:
        raise ValueError("No valid (non-NaN) binding energy rows found")
    return df


def formula_elements(formula):
    return set(re.findall(r"[A-Z][a-z]?", formula))


def formula_to_color(formula):
    present = formula_elements(formula)
    for element in HIGHLIGHT_ELEMENTS:
        if element in present:
            return HIGHLIGHT_COLORS[element]
    return DEFAULT_COLOR


def add_marginal_hist(ax, data, orientation, bin_range, n_bins=20):
    counts, bin_edges, _ = ax.hist(
        data, bins=n_bins, range=bin_range, density=True, orientation=orientation,
        color=HIST_COLOR, alpha=0.6, edgecolor="white",
    )
    if len(data) > 1:
        kde = gaussian_kde(data)
        x = np.linspace(bin_range[0], bin_range[1], 300)
        y = kde(x)
        if orientation == "vertical":
            ax.plot(x, y, color=KDE_COLOR, linewidth=1.5)
        else:
            ax.plot(y, x, color=KDE_COLOR, linewidth=1.5)
    ax.axis("off")


def plot_n_vs_o(df, output_path):
    n_E = df["n_computed_binding_E"].to_numpy() - N_SHIFT
    o_E = df["o_computed_binding_E"].to_numpy() - O_SHIFT
    colors = df["reduced_formula"].apply(formula_to_color).to_numpy()

    lo = min(n_E.min(), o_E.min()) - 0.2
    hi = max(n_E.max(), o_E.max()) + 0.2

    fig = plt.figure(figsize=(6, 6), dpi=500)
    gs = fig.add_gridspec(
        2, 2, width_ratios=(5, 1), height_ratios=(1, 5),
        left=0.12, right=0.98, bottom=0.1, top=0.98,
        wspace=0.03, hspace=0.03,
    )
    ax = fig.add_subplot(gs[1, 0])
    ax_top = fig.add_subplot(gs[0, 0], sharex=ax)
    ax_right = fig.add_subplot(gs[1, 1], sharey=ax)

    o_line = np.linspace(lo, hi, 100)
    n_left = LEFT_SLOPE * o_line + LEFT_INT
    n_right = RIGHT_SLOPE * o_line + RIGHT_INT

    ax.fill_between(o_line, n_left, n_right, color=REGION_COLOR, alpha=0.2, zorder=0)
    ax.plot(o_line, n_left, color=REGION_COLOR, linestyle="--", linewidth=1.2, zorder=1)
    ax.plot(o_line, n_right, color=REGION_COLOR, linestyle="--", linewidth=1.2, zorder=1)

    ax.scatter(o_E, n_E, color=colors, alpha=0.7, edgecolor="black", linewidth=0.5, s=100, zorder=2)

    # Legend handles for highlighted elements + "Other"
    handles = [
        plt.scatter([], [], color=HIGHLIGHT_COLORS[el], edgecolor="black", linewidth=0.5, s=100, label=el)
        for el in HIGHLIGHT_ELEMENTS
    ]
    handles.append(plt.scatter([], [], color=DEFAULT_COLOR, edgecolor="black", linewidth=0.5, s=100, label="Other"))
    handles.append(plt.Rectangle((0, 0), 1, 1, color=REGION_COLOR, alpha=0.2))

    ax.set_xlabel("O computed binding energy (eV)")
    ax.set_ylabel("N computed binding energy (eV)")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(handles=handles, frameon=False, fontsize=10, loc="upper left", ncol=1)

    add_marginal_hist(ax_top, o_E, "vertical", (lo, hi))
    add_marginal_hist(ax_right, n_E, "horizontal", (lo, hi))

    save_flat_svg(fig, output_path)
    plt.show()


file = "./data/ENR_volcano/n_o_matches.csv"
output_path = "./fig5/panel/n_vs_o_binding_energy.svg"

df = load_binding_energies(file)
plot_n_vs_o(df, output_path)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

BAR_COLOR = "#4C72B0"

plt.rcParams.update({"font.size": 14})


def save_flat_svg(fig, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    fig.savefig(path, format="svg", dpi=500, bbox_inches="tight")
    print(f"Saved plot to {path}")


def load_element_counts(file):
    df = pd.read_csv(file)
    for col in ("Element", "Count"):
        if col not in df.columns:
            raise KeyError(f"Expected column '{col}' not found in CSV")
    return df.sort_values("Count", ascending=False)


def plot_element_counts(df, output_path):
    fig, ax = plt.subplots(figsize=(12, 5), dpi=500)

    ax.bar(df["Element"], df["Count"], color=BAR_COLOR, edgecolor="black", linewidth=0.5)

    ax.set_xlabel("Element")
    ax.set_ylabel("Count")
    ax.spines[["top", "right"]].set_visible(False)
    plt.xticks(rotation=90)

    fig.tight_layout()
    #save_flat_svg(fig, output_path)
    plt.show()


file = "./data/ENR_volcano/n_o_element_count.csv"
output_path = "./fig5/panel/n_o_element_count.svg"

df = load_element_counts(file)
plot_element_counts(df, output_path)